# Bronze proof

Runs the real Bronze notebook with different parameters and shows the evidence the rubric asks for. Run after the first full load. Keep this notebook with its outputs (File > Export) in `proof/`.

### PROOF NOTEBOOK - Raw to Bronze (Phase 2 rubric)

In [ ]:
# PROOF NOTEBOOK - Raw to Bronze (Phase 2 rubric)
# Run this notebook top to bottom AFTER the first full load. Leave the outputs saved, or screen-record it.
# It drives the real Bronze notebook (./frontier_bronze_load) through its parameters. Nothing is faked.
# Parts: 1 schemas | 2 idempotent MERGE | 3 backfill | 4 schema drift | 4B batch range + replay from the start |
#        5 every run is logged | 6 log tables (primary key checks included)
# The drift test uses its own schema (bronze_test), so the real Bronze tables are never changed by it.

# CELL 1 - Setup
import hashlib, json, os, uuid
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "workspace", "catalog")
dbutils.widgets.text("bronze_notebook", "./frontier_bronze_load", "path of the Bronze notebook")
dbutils.widgets.dropdown("cleanup", "false", ["false", "true"], "cleanup: drop the bronze_test schema at the end")
CATALOG = dbutils.widgets.get("catalog").strip()
NB = dbutils.widgets.get("bronze_notebook").strip()
ROOT = f"/Volumes/{CATALOG}/landing/staging/frontier"
T_RUNS = f"{CATALOG}.ops.pipeline_execution_logs"
BRONZE = ["arena_leaderboard_raw", "cooperbench_member_raw", "benchlm_raw", "epoch_models_raw"]


def run_bronze(**params):
    """Runs the Bronze notebook with the given parameters. A failing run is printed, not hidden."""
    params = {k: str(v) for k, v in params.items()}
    print("PARAMETERS:", params)
    try:
        dbutils.notebook.run(NB, 3600, params)
        print("-> notebook finished without error")
    except Exception as exc:
        print("-> notebook ended with an error (expected in the failure test):", str(exc)[:300])


def counts(schema="bronze"):
    out = {}
    for t in BRONZE:
        try:
            out[t] = spark.table(f"{CATALOG}.{schema}.{t}").count()
        except Exception:
            out[t] = None
    return out


def manifests():
    folder = f"{ROOT}/processed/_manifests"
    found = []
    for f in dbutils.fs.ls(folder):
        with open(f.path.replace("dbfs:", ""), encoding="utf-8") as h:
            found.append(json.load(h))
    return sorted(found, key=lambda m: m["created_at"])


def last_runs(n=12, layer_like="Raw-to-Bronze%"):
    return spark.sql(f"""SELECT layer, source, dataset, file_or_param, status, start_time, end_time, rows_read,
        rows_inserted, rows_updated, rows_skipped_duplicate, rows_quarantined, message
        FROM {T_RUNS} WHERE layer LIKE '{layer_like}' ORDER BY load_timestamp DESC LIMIT {n}""")

### PROOF 1 - Explicit schemas: the tables carry the contract types, no column was inferred

In [ ]:
# PROOF 1 - Explicit schemas: the tables carry the contract types, no column was inferred
for t in BRONZE:
    print("=====", t)
    display(spark.sql(f"DESCRIBE TABLE {CATALOG}.bronze.{t}"))
print("The StructType contracts themselves are in frontier_bronze_load: ARENA_CONTRACT, EPOCH_CONTRACT,")
print("COOP_ROW_SCHEMA and BLM_ROW_SCHEMA (Cell 2). `grep -n inferSchema` finds nothing in the repo.")

### PROOF 2 - MERGE INTO is idempotent: the same input twice changes nothing

In [ ]:
# PROOF 2 - MERGE INTO is idempotent: the same input twice changes nothing
ms = manifests()
assert ms, "No processed batch yet: run the full load first."
batch = ms[-1]["batch_id"]
before = counts()
run_bronze(source_dir="processed", batch_id=batch, force_reload="true")      # force = really re-run MERGE
after_1 = counts()
run_bronze(source_dir="processed", batch_id=batch, force_reload="true")      # and once more
after_2 = counts()
print("rows before :", before)
print("rows after 1:", after_1)
print("rows after 2:", after_2)
assert before == after_1 == after_2, "Row counts changed: the MERGE is not idempotent!"
print("OK - identical row counts, no duplicates created.")
for t in BRONZE:
    dup = spark.sql(f"SELECT COUNT(*) c FROM (SELECT record_hash FROM {CATALOG}.bronze.{t} "
                    f"GROUP BY record_hash HAVING COUNT(*) > 1)").first()["c"]
    print(f"duplicate record_hash in {t}: {dup}")
display(last_runs(10))          # rows_inserted = 0 and rows_skipped_duplicate = rows_read for the re-runs

### PROOF 3 - Backfill of an older period, only parameters, no code change

In [ ]:
# PROOF 3 - Backfill of an older period, only parameters, no code change
first_day = ms[0]["created_at"][:10]
before = counts()
run_bronze(source_dir="processed", start_date=first_day, end_date=first_day, force_reload="true")   # by date
run_bronze(source_dir="processed", batch_id=ms[0]["batch_id"], source="arena", force_reload="true") # batch + source
print("rows before:", before)
print("rows after :", counts())
display(last_runs(12))

### PROOF 4 - Schema drift: a new column evolves the table, a type change is quarantined, the batch still finishes

In [ ]:
# PROOF 4 - Schema drift: a new column evolves the table, a type change is quarantined, the batch still finishes
import numpy as np, pyarrow as pa, pyarrow.parquet as pq

TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
TEST_DIR = f"{ROOT}/drift_test"
BATCH = f"{TAG}_drifttest_incremental"
N = 60_000                                  # about 3 MB per file, so the 1 MB incremental rule is met honestly
rng = np.random.default_rng(42)


def arena_table(rating_as_text=False, new_column=False):
    cols = {"model_name": [f"drift-test-{TAG}-{i}" for i in range(N)], "organization": ["test-org"] * N,
            "license": ["test"] * N,
            "rating": [str(x) for x in rng.random(N) * 1000] if rating_as_text else rng.random(N) * 1000,
            "variance": rng.random(N), "rating_q975": rng.random(N) * 1000, "rating_q025": rng.random(N) * 1000,
            "vote_count": rng.integers(0, 10**6, N), "rank": np.arange(N), "category": ["overall"] * N,
            "leaderboard_publish_date": ["2026-10-01"] * N}
    if new_column:
        cols["new_metric"] = rng.random(N)
    return pa.table(cols)


files = []
for label, table in [("A_new_column", arena_table(new_column=True)), ("B_type_change", arena_table(rating_as_text=True))]:
    rel = f"arena/text__incremental__DRIFT_{label}_{TAG}.parquet"
    tmp = f"/tmp/{os.path.basename(rel)}"
    pq.write_table(table, tmp)
    dbutils.fs.mkdirs(f"{TEST_DIR}/arena")
    dbutils.fs.cp(f"file:{tmp}", f"{TEST_DIR}/{rel}", True)
    h = hashlib.sha256(open(tmp, "rb").read()).hexdigest()
    size = os.path.getsize(tmp)
    files.append({"key": f"arena/text/{label}", "source": "arena", "dataset": "text", "path": rel,
                  "size_bytes": size, "sha256": h, "fingerprint": None, "landed_bytes": size})
manifest = {"batch_id": BATCH, "load_type": "incremental", "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "required_bytes": 1_000_000, "downloaded_bytes": sum(f["size_bytes"] for f in files),
            "landed_bytes": sum(f["landed_bytes"] for f in files), "gate_passed": True, "files": files}
dbutils.fs.mkdirs(f"{TEST_DIR}/_manifests")
with open(f"{TEST_DIR}/_manifests/manifest__{BATCH}.json", "w", encoding="utf-8") as h:
    json.dump(manifest, h, indent=2)

# File A adds the column new_metric (table evolves). File B has rating as TEXT (type change -> quarantined).
run_bronze(source_dir=TEST_DIR, batch_id=BATCH, bronze_schema="bronze_test", source="arena")

T_TEST = f"{CATALOG}.bronze_test.arena_leaderboard_raw"
print("Columns of the test table now include new_metric:", "new_metric" in spark.table(T_TEST).columns)
print("Rows loaded from file A:", spark.table(T_TEST).count(), "(file B was quarantined, not loaded)")
display(spark.sql(f"""SELECT detected_at, source_file, column_name, event_type, expected_type, actual_type, action_taken
    FROM {CATALOG}.ops.schema_drift_log WHERE batch_id = '{BATCH}' ORDER BY detected_at"""))
display(spark.sql(f"""SELECT source_file, reason, detail, record_sample, quarantined_at
    FROM {CATALOG}.ops.quarantine_log WHERE batch_id = '{BATCH}'"""))
display(spark.sql(f"""SELECT file_or_param, status, rows_read, rows_inserted, rows_quarantined, message
    FROM {T_RUNS} WHERE batch_id = '{BATCH}' ORDER BY load_timestamp"""))

### PROOF 4B - Backfill from ANY batch to ANY batch, and a replay from the first batch, batch by batch.

In [ ]:
# PROOF 4B - Backfill from ANY batch to ANY batch, and a replay from the first batch, batch by batch.
# Three small test batches arrive on three different "days". Batches 1-2 and 2-3 share some rows on purpose,
# so the replay has to deal with rows it has already seen. Everything goes to its own test schema.
RTAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RDIR = f"{ROOT}/replay_test"
RSCHEMA = "bronze_replay_test"
M = 150_000
R = np.random.default_rng(1).random((M, 4))            # fixed numbers per row index: same row = same content


def replay_rows(first, last):
    i = np.arange(first, last)
    return pa.table({"model_name": [f"replay-{RTAG}-{k}" for k in i], "organization": ["test-org"] * len(i),
                     "license": ["test"] * len(i), "rating": R[i, 0] * 1000, "variance": R[i, 1],
                     "rating_q975": R[i, 2] * 1000, "rating_q025": R[i, 3] * 1000, "vote_count": i * 3,
                     "rank": i, "category": ["overall"] * len(i), "leaderboard_publish_date": ["2026-10-01"] * len(i)})


RANGES = {1: (0, 60_000), 2: (30_000, 90_000), 3: (60_000, 120_000)}     # 120,000 different rows in total
dbutils.fs.mkdirs(f"{RDIR}/arena")
dbutils.fs.mkdirs(f"{RDIR}/_manifests")
RBATCHES = {}
for n, (a, b) in RANGES.items():
    rel = f"arena/text__incremental__REPLAY_{n}_{RTAG}.parquet"
    tmp = f"/tmp/{os.path.basename(rel)}"
    pq.write_table(replay_rows(a, b), tmp)
    dbutils.fs.cp(f"file:{tmp}", f"{RDIR}/{rel}", True)
    size = os.path.getsize(tmp)
    bid = f"2026100{n}T030000Z_{RTAG}_incremental"
    RBATCHES[n] = bid
    man = {"batch_id": bid, "load_type": "incremental", "created_at": f"2026-10-0{n}T03:00:00Z",
           "required_bytes": 1_000_000, "downloaded_bytes": size, "landed_bytes": size, "gate_passed": True,
           "files": [{"key": f"arena/text/replay{n}", "source": "arena", "dataset": "text", "path": rel,
                      "size_bytes": size, "sha256": hashlib.sha256(open(tmp, "rb").read()).hexdigest(),
                      "fingerprint": None, "landed_bytes": size}]}
    with open(f"{RDIR}/_manifests/manifest__{bid}.json", "w", encoding="utf-8") as h:
        json.dump(man, h, indent=2)
T_RT = f"{CATALOG}.{RSCHEMA}.arena_leaderboard_raw"
common = dict(source_dir=RDIR, bronze_schema=RSCHEMA, source="arena")


def rt_rows():
    return spark.table(T_RT).count() if spark.catalog.tableExists(T_RT) else 0


# (a) Backfill of ONE batch out of the middle: only batch 2 is loaded
run_bronze(from_batch=RBATCHES[2], to_batch=RBATCHES[2], **common)
print("rows after loading only batch 2:", rt_rows(), "(expected 60000)")

# (b) Replay from the first batch: batches 1, 2, 3 are loaded one by one, oldest first
run_bronze(replay="true", **common)
print("rows after replay:", rt_rows(), "(expected 120000: the shared rows were not duplicated)")

# (c) Replay again: nothing changes
run_bronze(replay="true", **common)
assert rt_rows() == 120_000, "A second replay changed the row count!"

# (d) Rebuild: empty the table, replay again. The data comes back batch by batch as it arrived, no full load.
spark.sql(f"TRUNCATE TABLE {T_RT}")
run_bronze(replay="true", **common)
assert rt_rows() == 120_000, "Rebuild did not give the same rows!"
print("OK - rebuilt from the first batch: 120000 rows, identical to before.")

# (e) How each batch behaved in the last replay (oldest first), and the primary key checks
display(spark.sql(f"""SELECT start_time, layer, file_or_param, status, rows_read, rows_inserted,
    rows_skipped_duplicate, message FROM {T_RUNS}
    WHERE batch_id LIKE '%{RTAG}%' AND layer IN ('Raw-to-Bronze', 'Raw-to-Bronze (batch)')
    ORDER BY start_time DESC LIMIT 12"""))
display(spark.sql(f"""SELECT start_time, layer, dataset, status, rows_read, message FROM {T_RUNS}
    WHERE layer = 'Bronze data quality (primary key)' ORDER BY start_time DESC LIMIT 8"""))

### PROOF 5 - Every run is logged: a run with nothing to do, and a run with a bad parameter

In [ ]:
# PROOF 5 - Every run is logged: a run with nothing to do, and a run with a bad parameter
run_bronze(source_dir="processed", batch_id="no_such_batch")                  # no batch matches -> SUCCESS row, 0 files
run_bronze(source_dir="processed", start_date="2026-13-45")                   # invalid date -> FAILURE row, no crash before logging
display(last_runs(4, "Raw-to-Bronze (run)"))

### PROOF 6 - The log table (all layers) and the error table

In [ ]:
# PROOF 6 - The log table (all layers) and the error table
display(spark.sql(f"SELECT * FROM {T_RUNS} ORDER BY load_timestamp DESC LIMIT 30"))
display(spark.sql(f"""SELECT occurred_at, component, severity, source, file_or_param, error_type, error_message
    FROM {CATALOG}.ops.error_logs ORDER BY occurred_at DESC LIMIT 15"""))

### CELL 7 (optional) - remove the test schema and test files. Set the `cleanup` widget to true first.

In [ ]:
# CELL 7 (optional) - remove the test schema and test files. Set the `cleanup` widget to true first.
if dbutils.widgets.get("cleanup") == "true":
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze_test CASCADE")
    spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze_replay_test CASCADE")
    dbutils.fs.rm(TEST_DIR, True)
    dbutils.fs.rm(RDIR, True)
    print("bronze_test, bronze_replay_test and the test files were removed.")
else:
    print("Cleanup skipped (the log rows in ops.* stay either way, they are the proof).")